# 03 — News context (hardened)

Pulls neutral, non-sensational news context using:
- NewsAPI (if `NEWS_API_KEY` present)
- NewsData.io (if `NEWS_DATA_IO_KEY` present)
- Otherwise falls back to GDELT (no key)

This replacement notebook is hardened to:
- tolerate malformed provider responses
- record provider failures instead of crashing
- preserve provenance for both successes and failures

Outputs:
- `outputs/03_news_context/news_articles.json`
- `outputs/03_news_context/news_provider_failures.csv`
- `outputs/03_news_context/manifest.json`

In [ ]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
import yaml
import pandas as pd

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_file(p: Path) -> str:
    return sha256_bytes(p.read_bytes())

def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def manifest_base(step: str, config_paths=None):
    m = {
        "step": step,
        "created_at_utc": utc_now(),
        "configs": [],
        "artifacts": [],
        "provider_failures": [],
    }
    for cp in (config_paths or []):
        if cp and cp.exists():
            m["configs"].append({"path": str(cp), "sha256": sha256_file(cp)})
    return m

def add_artifact(manifest: dict, path: Path, kind: str, meta=None):
    if path.exists():
        item = {"kind": kind, "path": str(path), "sha256": sha256_file(path)}
        if meta:
            item["meta"] = meta
        manifest["artifacts"].append(item)

def env(name: str) -> str | None:
    v = os.getenv(name)
    return v.strip() if isinstance(v, str) and v.strip() else None

def safe_json_response(resp):
    try:
        return resp.json(), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

def record_failure(failures, provider: str, query: str, resp=None, reason: str = ""):
    row = {
        "provider": provider,
        "query": query,
        "reason": reason,
        "status_code": getattr(resp, "status_code", None),
        "ok": getattr(resp, "ok", None),
        "content_type": (getattr(resp, "headers", {}) or {}).get("Content-Type", ""),
        "response_preview": (getattr(resp, "text", "") or "")[:500],
    }
    failures.append(row)

# --- resolve repo root robustly ---
cwd = Path.cwd().resolve()

def find_repo_root(start: Path) -> Path:
    cur = start
    for _ in range(8):
        if (cur / "configs").exists() and (cur / "notebooks").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start

ROOT = find_repo_root(cwd)
CONFIGS = ROOT / "configs"
OUTPUTS = ROOT / "outputs"

print("CWD:", cwd)
print("ROOT:", ROOT)

In [ ]:
step = "03_news_context"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "run.yml"
if not cfg_path.exists():
    raise FileNotFoundError("configs/run.yml not found")

cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}

news_cfg = cfg.get("news") or {}
terms = news_cfg.get("query_terms")

if not terms:
    sites = cfg.get("sites")
    derived = []
    if isinstance(sites, list):
        for s in sites:
            nm = (s.get("name") or "").strip()
            if nm:
                derived.append(nm)
    elif isinstance(sites, dict):
        for k, v in sites.items():
            nm = (v.get("name") or k or "").strip()
            if nm:
                derived.append(nm)

    base_terms = [
        "Newhaven incinerator",
        "Newhaven Energy Recovery Facility",
        "Lewes air quality",
        "Brighton air quality",
        "Eastbourne air quality",
        "PM2.5 Sussex",
        "NO2 Sussex",
        "wood burning smoke UK",
        "temperature inversion UK",
    ]

    combined = derived + base_terms
    seen = set()
    terms = []
    for t in combined:
        t = t.strip()
        if not t:
            continue
        key = t.lower()
        if key in seen:
            continue
        seen.add(key)
        terms.append(t)

max_records = int(news_cfg.get("max_records", 25))
recency_days = int(news_cfg.get("recency_days", 30))

print("Query terms:", len(terms))
for t in terms[:10]:
    print(" -", t)
print("Max records:", max_records, "Recency days:", recency_days)

In [ ]:
newsapi_key = env("NEWS_API_KEY")
newsdata_key = env("NEWS_DATA_IO_KEY")

articles = []
provider_failures = []

def dedupe(arts):
    seen = set()
    out_rows = []
    for a in arts:
        u = a.get("url")
        if u and u in seen:
            continue
        if u:
            seen.add(u)
        out_rows.append(a)
    return out_rows

# --- NewsAPI ---
if newsapi_key:
    for term in terms:
        try:
            r = requests.get(
                "https://newsapi.org/v2/everything",
                params={
                    "q": term,
                    "pageSize": min(10, max_records),
                    "sortBy": "publishedAt",
                    "language": "en",
                },
                headers={"X-Api-Key": newsapi_key},
                timeout=30,
            )
            payload, json_err = safe_json_response(r)
            if r.ok and payload is not None:
                for a in payload.get("articles", []):
                    articles.append({
                        "provider": "newsapi",
                        "query": term,
                        "title": a.get("title"),
                        "source": (a.get("source") or {}).get("name"),
                        "publishedAt": a.get("publishedAt"),
                        "url": a.get("url"),
                        "description": a.get("description"),
                    })
            else:
                record_failure(provider_failures, "newsapi", term, r, json_err or "non-ok response")
        except Exception as e:
            record_failure(provider_failures, "newsapi", term, None, f"{type(e).__name__}: {e}")

# --- NewsData.io ---
if newsdata_key:
    for term in terms:
        try:
            r = requests.get(
                "https://newsdata.io/api/1/news",
                params={
                    "apikey": newsdata_key,
                    "q": term,
                    "language": "en",
                    "country": "gb",
                },
                timeout=30,
            )
            payload, json_err = safe_json_response(r)
            if r.ok and payload is not None:
                for a in payload.get("results", [])[: min(10, max_records)]:
                    articles.append({
                        "provider": "newsdataio",
                        "query": term,
                        "title": a.get("title"),
                        "source": a.get("source_id"),
                        "publishedAt": a.get("pubDate"),
                        "url": a.get("link"),
                        "description": a.get("description"),
                    })
            else:
                record_failure(provider_failures, "newsdataio", term, r, json_err or "non-ok response")
        except Exception as e:
            record_failure(provider_failures, "newsdataio", term, None, f"{type(e).__name__}: {e}")

# --- GDELT fallback (no key) ---
if not newsapi_key and not newsdata_key:
    for term in terms:
        try:
            r = requests.get(
                "https://api.gdeltproject.org/api/v2/doc/doc",
                params={
                    "query": term,
                    "mode": "ArtList",
                    "format": "json",
                    "maxrecords": min(10, max_records),
                },
                timeout=30,
            )
            payload, json_err = safe_json_response(r)
            if r.ok and payload is not None:
                for a in payload.get("articles", []):
                    articles.append({
                        "provider": "gdelt",
                        "query": term,
                        "title": a.get("title"),
                        "source": a.get("sourceCountry"),
                        "publishedAt": a.get("seendate"),
                        "url": a.get("url"),
                        "description": a.get("summary"),
                    })
            else:
                record_failure(provider_failures, "gdelt", term, r, json_err or "non-ok response")
        except Exception as e:
            record_failure(provider_failures, "gdelt", term, None, f"{type(e).__name__}: {e}")

deduped = dedupe(articles)
write_json(out / "news_articles.json", deduped)

fail_df = pd.DataFrame(provider_failures)
fail_df.to_csv(out / "news_provider_failures.csv", index=False)

manifest = manifest_base(step, config_paths=[cfg_path])
add_artifact(
    manifest,
    out / "news_articles.json",
    "news_metadata",
    meta={
        "count": len(deduped),
        "providers": sorted({a.get('provider') for a in deduped if a.get('provider')}),
    },
)
add_artifact(
    manifest,
    out / "news_provider_failures.csv",
    "provider_failures",
    meta={"count": len(provider_failures)},
)
manifest["provider_failures"] = provider_failures[:50]
write_json(out / "manifest.json", manifest)

print("Articles:", len(deduped))
print("Provider failures:", len(provider_failures))
for a in deduped[:10]:
    print("-", a.get("publishedAt"), a.get("source"), a.get("title"))
if provider_failures:
    print("\nFirst provider failure:")
    print(provider_failures[0])
print("Wrote:", out / "manifest.json")